In [1]:
# Let's install the tools we'll need for our project
!pip install "transformers" "datasets" "accelerate" "peft" "bitsandbytes" "transformer_lens" "trl"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The specific Qwen model we'll use
model_id = "Qwen/Qwen1.5-0.5B-Chat"

# This is our QLoRA configuration.
# It tells the transformer to load the model in 4-bit precision, which saves a ton of memory.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the model from Hugging Face with our special config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # This automatically puts the model on the GPU if it's available
)

# Load the tokenizer that matches the model
tokenizer = AutoTokenizer.from_pretrained(model_id)

# A quick check to make sure everything is ready
print("✅ Model and tokenizer loaded successfully!")

In [ ]:
from transformer_lens import HookedTransformer

print("\n" + "="*50)
print("🚀 Running the 'BEFORE' benchmark on the base model ...")
print("="*50)

# We go back to the manual HookedTransformer setup that worked before
hooked_model = HookedTransformer.from_pretrained(model_id)
hooked_model.eval()

# Our test prompt
# test_prompt = "Implement a basic Stack class with push, pop, and peek methods"
test_prompt="""
Write a Python program that shows 20 balls bouncing inside a spinning hexagon:
- All balls have the same radius.
- All balls have a number on it from 1 to 20.
- All balls drop from the heptagon center when starting.
- Colors are: #f8b862, #f6ad49, #f39800, #f08300, #ec6d51, #ee7948, #ed6d3d, #ec6800, #ec6800, #ee7800, #eb6238, #ea5506, #ea5506, #eb6101, #e49e61, #e45e32, #e17b34, #dd7a56, #db8449, #d66a35
- The balls should be affected by gravity and friction, and they must bounce off the rotating walls realistically. There should also be collisions between balls.
- The material of all the balls determines that their impact bounce height will not exceed the radius of the heptagon, but higher than ball radius.
- All balls rotate with friction, the numbers on the ball can be used to indicate the spin of the ball.
- The heptagon is spinning around its center, and the speed of spinning is 360 degrees per 5 seconds.
- The heptagon size should be large enough to contain all the balls.
- Do not use the pygame library; implement collision detection algorithms and collision response etc. by yourself. The following Python libraries are allowed: tkinter, math, numpy, dataclasses, typing, sys.
- All codes should be put in a single Python file.

"""
chat_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": test_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# --- STEP 1: Generate Text using the standard Hugging Face method ---
inputs = tokenizer(chat_prompt, return_tensors="pt")
generated_ids = model.generate(inputs.input_ids, max_new_tokens=150, do_sample=False)
base_model_output_text = tokenizer.decode(generated_ids[0])

print("\n--- Base Model Output ---\n")
print(base_model_output_text)
print("\n-------------------------\n")


# --- STEP 2: Analyze by running the full text through run_with_cache ---
print("Analyzing the generation process...")
# We use the full output text to get the cache for the *entire* sequence.
# The first return value is the logits, which we can ignore with '_'
_, base_model_cache = hooked_model.run_with_cache(base_model_output_text)

print("✅ 'BEFORE' benchmark complete. Activations are now saved in 'base_model_cache'.")

In [ ]:
from datasets import load_dataset

# Load the dataset from Hugging Face
# We're taking the 'train' split which has the most data
dataset = load_dataset("deepmind/code_contests", split="train")


# For our first experiment, let's shuffle and pick a smaller subset of 1000 examples
# This will make the fine-tuning process much faster
dataset = dataset.shuffle(seed=42).select(range(1000))

# This function takes a row from the dataset and formats it into the chat template
def format_chat_template(row):
    # We only want to train on problems that have a Python solution
    if "python" not in row["solutions"] or not row["solutions"]["python"]:
        return None # This will be filtered out

    # We use the problem 'description' as the user's prompt
    # and the first Python solution as the assistant's ideal response.
    # The tokenizer handles putting the special tokens in the right places.
    formatted_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "You are a helpful assistant that solves coding problems."},
            {"role": "user", "content": row["description"]},
            {"role": "assistant", "content": row["solutions"]["python"][0]},
        ],
        tokenize=False,
    )
    return {"text": formatted_text}

# Apply the formatting function to our dataset
# We use 'map' to apply our function to every example in the dataset
# and then filter out any examples that didn't have a Python solution
original_columns = dataset.column_names
dataset = dataset.map(format_chat_template, remove_columns=original_columns).filter(lambda x: x is not None)


# Let's check our work
print("✅ Dataset prepared successfully!")
print("\nHere is an example of a single formatted data point:\n")
print(dataset[0])

In [ ]:
from transformers import TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# Define a formatting function that processes batches of examples.
# This ensures that the 'text' field contains plain strings, not nested structures.
def formatting_func(examples):
    descs = examples["description"]  # Extract the descriptions from the batch
    texts = []
    for desc in descs:
        if isinstance(desc, list):
            # If description is a list, join into a single string
            text = " ".join(desc)
        elif isinstance(desc, dict):
            # If description is a dict, extract relevant text (adjust as needed)
            text = desc.get("text", "")
        else:
            # Otherwise, convert to string directly
            text = str(desc)
        texts.append(text)
    return {"text": texts}

# Apply the formatting function to the dataset.
# It is essential to assign the result back to the dataset to update it properly.
dataset = dataset.map(formatting_func, batched=True)

# LoRA configuration for parameter-efficient fine-tuning.
lora_config = LoraConfig(
    r=8,  # Rank; complexity of low-rank layers added
    lora_alpha=32,  # Scaling factor for LoRA weights
    target_modules=["q_proj", "v_proj"],  # Target modules for LoRA modification
    lora_dropout=0.05,  # Dropout to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM",  # Task type for causal language modeling
)

# Training arguments controlling training behavior.
training_args = TrainingArguments(
    output_dir="./qwen-sft-finetuned",  # Directory for saving outputs
    num_train_epochs=1,  # Number of epochs to train
    per_device_train_batch_size=2,  # Batch size per device
    gradient_accumulation_steps=4,  # To simulate larger batch size
    learning_rate=2e-4,  # Learning rate
    logging_steps=10,  # Logging frequency
    optim="paged_adamw_8bit",  # Optimizer with memory efficiency
    save_strategy="epoch",  # Save model every epoch
)

# Create the trainer.
# NOTE: Do NOT pass 'formatting_func' here again to avoid double application and nested dict errors.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
)

# Start training.
print("🚀 Starting the fine-tuning process... this will take some time!")
trainer.train()
print("✅ Fine-tuning complete!")

# Save the final fine-tuned model adapters.
final_model_path = "./qwen_sft_final"
trainer.save_model(final_model_path)
print(f"Final model saved to {final_model_path}")


OPTIMISED CODE

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The specific Qwen model we'll use
model_id = "Qwen/Qwen1.5-0.5B-Chat"

# This is our QLoRA configuration.
# It tells the transformer to load the model in 4-bit precision, which saves a ton of memory.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the model from Hugging Face with our special config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # This automatically puts the model on the GPU if it's available
)

# Load the tokenizer that matches the model
tokenizer = AutoTokenizer.from_pretrained(model_id)

# A quick check to make sure everything is ready
print("✅ Model and tokenizer loaded successfully!")


# ================================
# == OPTIMIZATION SUMMARY ==
# No changes were made to the code as it already uses the most important memory-saving technique:
#
# 1.  `load_in_4bit=True`: This loads the model weights using only 4 bits per parameter instead of the usual 32 or 16.
#     This reduces the model's memory footprint by up to 8x.
# 2.  `bnb_4bit_compute_dtype=torch.bfloat16`: While the model is stored in 4-bit, computations (like matrix multiplications)
#     are performed in 16-bit for accuracy and stability. This is a good trade-off between performance and memory.
# 3.  `device_map="auto"`: This intelligently places the model parts on the available hardware (GPU/CPU), which is essential for making the most of a free-tier GPU.
# ================================

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model and tokenizer loaded successfully!


In [14]:
from transformer_lens import HookedTransformer
import gc

print("\n" + "="*50)
print("🚀 Running the 'BEFORE' benchmark on the base model ...")
print("="*50)

# We load the HookedTransformer, but we'll delete it right after use
# to free up memory for fine-tuning.
# NOTE: This still temporarily loads a full-precision copy of the model.
# On extremely constrained systems, you might need to skip this analysis step entirely.
hooked_model = HookedTransformer.from_pretrained(model_id)
hooked_model.to("cuda" if torch.cuda.is_available() else "cpu")
hooked_model.eval()

# Our test prompt
test_prompt="""
Write a Python program that shows 20 balls bouncing inside a spinning hexagon:
- All balls have the same radius.
- All balls have a number on it from 1 to 20.
- All balls drop from the heptagon center when starting.
- Colors are: #f8b862, #f6ad49, #f39800, #f08300, #ec6d51, #ee7948, #ed6d3d, #ec6800, #ec6800, #ee7800, #eb6238, #ea5506, #ea5506, #eb6101, #e49e61, #e45e32, #e17b34, #dd7a56, #db8449, #d66a35
- The balls should be affected by gravity and friction, and they must bounce off the rotating walls realistically. There should also be collisions between balls.
- The material of all the balls determines that their impact bounce height will not exceed the radius of the heptagon, but higher than ball radius.
- All balls rotate with friction, the numbers on the ball can be used to indicate the spin of the ball.
- The heptagon is spinning around its center, and the speed of spinning is 360 degrees per 5 seconds.
- The heptagon size should be large enough to contain all the balls.
- Do not use the pygame library; implement collision detection algorithms and collision response etc. by yourself. The following Python libraries are allowed: tkinter, math, numpy, dataclasses, typing, sys.
- All codes should be put in a single Python file.
"""
chat_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": test_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# --- STEP 1: Generate Text using the standard Hugging Face method ---
# We use our memory-efficient 4-bit model for this generation
inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
generated_ids = model.generate(
    inputs.input_ids,
    do_sample=True,          # Enable sampling for top_p to work
    top_p=0.9,
    temperature=0.7,
    max_new_tokens=1000,     # Increased to allow full code generation
    repetition_penalty=1.1
)
base_model_output_text = tokenizer.decode(generated_ids[0])

print("\n--- Base Model Output ---\n")
print(base_model_output_text)
print("\n-------------------------\n")


# --- STEP 2: Analyze by running the full text through run_with_cache ---
print("Analyzing the generation process...")
with torch.no_grad(): # Ensure no gradients are computed
    _, base_model_cache = hooked_model.run_with_cache(base_model_output_text)

print("✅ 'BEFORE' benchmark complete.")

# --- OPTIMIZATION: Explicitly delete the large objects and clear CUDA cache ---
print("Cleaning up memory before fine-tuning...")
del hooked_model
del base_model_cache
gc.collect()
torch.cuda.empty_cache()
print("✅ Memory cleaned.")


# ================================
# == OPTIMIZATION SUMMARY ==
# 1.  `Sequential Operation`: The original code kept two models in memory simultaneously. This version runs the
#     `HookedTransformer` analysis and then immediately deletes the objects.
# 2.  `del hooked_model, del base_model_cache`: We explicitly tell Python to remove our references to these large objects.
#     The `hooked_model` is a full-precision model, and `base_model_cache` contains all the activations, so they are both very large.
# 3.  `gc.collect()`: This forces Python's garbage collector to run, reclaiming memory from the deleted objects.
# 4.  `torch.cuda.empty_cache()`: This releases all unused cached memory on the GPU that PyTorch was holding onto.
#     This sequence ensures that the maximum amount of VRAM is available for the most demanding step: fine-tuning.
# ================================

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_posi


🚀 Running the 'BEFORE' benchmark on the base model ...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_posi

Loaded pretrained model Qwen/Qwen1.5-0.5B-Chat into HookedTransformer
Moving model to device:  cuda

--- Base Model Output ---

<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user

Write a Python program that shows 20 balls bouncing inside a spinning hexagon:
- All balls have the same radius.
- All balls have a number on it from 1 to 20.
- All balls drop from the heptagon center when starting.
- Colors are: #f8b862, #f6ad49, #f39800, #f08300, #ec6d51, #ee7948, #ed6d3d, #ec6800, #ec6800, #ee7800, #eb6238, #ea5506, #ea5506, #eb6101, #e49e61, #e45e32, #e17b34, #dd7a56, #db8449, #d66a35
- The balls should be affected by gravity and friction, and they must bounce off the rotating walls realistically. There should also be collisions between balls.
- The material of all the balls determines that their impact bounce height will not exceed the radius of the heptagon, but higher than ball radius.
- All balls rotate with friction, the numbers on the ball can be used to i

In [4]:
from datasets import load_dataset
import sys

print("🚀 Starting dataset preparation...")
dataset = load_dataset("deepmind/code_contests", split="train")
dataset = dataset.shuffle(seed=42).select(range(5000))

# Inspecting language IDs in a few examples to identify Python
all_langs = []
for i in range(20):
    langs = dataset[i]["solutions"]["language"]
    all_langs.extend(langs)
unique_langs = sorted(set(all_langs))
print("Unique language IDs in sample:", unique_langs)

language_examples = {}
for i in range(20):
    sol = dataset[i]["solutions"]
    for lang, code in zip(sol["language"], sol["solution"]):
        if lang not in language_examples:
            language_examples[lang] = []
        if len(language_examples[lang]) < 3:
            language_examples[lang].append(code)
for lang, codes in language_examples.items():
    print(f"Lang {lang} example: {repr(codes[0][:80])} ...")

# Manually set this after inspecting output above, e.g., 1 for Python
PYTHON_LANG_ID = 1


def format_chat_template(examples):
    texts = []
    for i in range(len(examples["description"])):
        sol = examples["solutions"][i]
        langs = sol["language"]
        codes = sol["solution"]
        # Extract Python solutions based on language ID
        py_solutions = [code for lang, code in zip(langs, codes) if lang == PYTHON_LANG_ID]
        if py_solutions:
            formatted_text = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": "You are a helpful assistant that solves coding problems."},
                    {"role": "user", "content": examples["description"][i]},
                    {"role": "assistant", "content": py_solutions[0]},
                ],
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(formatted_text + tokenizer.eos_token)
        else:
            texts.append(None)
    return {"text": texts}


print("\nFormatting and filtering examples...")
original_columns = dataset.column_names
dataset = dataset.map(format_chat_template, batched=True, remove_columns=original_columns)
dataset = dataset.filter(lambda example: example["text"] is not None)

# Limit dataset size for efficient training
DESIRED_SIZE = 1000
if len(dataset) > DESIRED_SIZE:
    dataset = dataset.select(range(DESIRED_SIZE))

if len(dataset) == 0:
    print("❌ No valid Python solutions found.")
    sys.exit()

print(f"✅ Dataset prepared with {len(dataset)} examples.")

🚀 Starting dataset preparation...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00039-e991a271dbfa99(…):   0%|          | 0.00/180M [00:00<?, ?B/s]

...

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

Unique language IDs in sample: [1, 2, 3, 4]
Lang 4 example: 'import java.io.ByteArrayInputStream;\nimport java.io.IOException;\nimport java.io.' ...
Lang 3 example: '# cook your dish here\n#code\nimport math\nimport collections\nfrom sys import stdin' ...
Lang 2 example: '#include <bits/stdc++.h>\nusing namespace std;\nconst double pi = acos(-1.0);\ncons' ...
Lang 1 example: 'import sys\ninput = sys.stdin.readline\nT = int(input())\n\ndef solve(m, n, arr):\n  ' ...

Formatting and filtering examples...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

✅ Dataset prepared with 1000 examples.


In [5]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from transformers.utils import logging


# Enable verbose logging including progress bar and detailed info
logging.set_verbosity_info()


class VerboseCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        loss_val = None
        if state.log_history and 'loss' in state.log_history[-1]:
            loss_val = state.log_history[-1]['loss']
        print(f"[Step {state.global_step}/{state.max_steps}] Loss: {loss_val}")


# LoRA parameter-efficient fine-tuning config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Supervised Fine-Tuning config with frequent logging
sft_config = SFTConfig(
    dataset_text_field="text",
    output_dir="./qwen-sft-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_strategy="steps",
    logging_steps=1,        # Log every training step for maximum visibility
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    fp16=True,
    report_to=[],           # Disable wandb logging prompts
)

# Instantiate trainer with verbose callback
trainer = SFTTrainer(
    model=model,                 # your preloaded Qwen model
    train_dataset=dataset,       # from Cell 1
    peft_config=lora_config,
    args=sft_config,
    callbacks=[VerboseCallback()]  # attach callback instance (note parentheses)
)

print("🚀 Starting supervised fine-tuning...")
trainer.train()
print("✅ Training complete!")

# Save LoRA adapters after training
final_model_path = "./qwen_sft_final"
trainer.save_model(final_model_path)
print(f"✅ LoRA adapters saved to: {final_model_path}")

PyTorch: setting up devices
average_tokens_across_devices is True but world size is 1. Setting it to False automatically.
loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabula

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Using auto half precision backend


...

[Step 2/250] Loss: 3.6237
[Step 3/250] Loss: 2.9607
[Step 4/250] Loss: 2.4636
[Step 5/250] Loss: 2.6907
[Step 6/250] Loss: 1.9262
[Step 7/250] Loss: 2.0152
[Step 8/250] Loss: 1.5816
[Step 9/250] Loss: 2.2529
[Step 10/250] Loss: 1.8585
[Step 11/250] Loss: 1.6206
[Step 12/250] Loss: 1.9371
[Step 13/250] Loss: 1.9205
[Step 14/250] Loss: 1.9924
[Step 15/250] Loss: 1.7563
[Step 16/250] Loss: 1.8043
[Step 17/250] Loss: 1.8893
[Step 18/250] Loss: 2.1608
[Step 19/250] Loss: 2.0114
[Step 20/250] Loss: 1.8224
[Step 21/250] Loss: 1.7678
[Step 22/250] Loss: 1.5537
[Step 23/250] Loss: 1.6985
[Step 24/250] Loss: 2.1321
[Step 25/250] Loss: 1.7359
[Step 26/250] Loss: 1.4622
[Step 27/250] Loss: 1.7241
[Step 28/250] Loss: 1.846
[Step 29/250] Loss: 1.8123
[Step 30/250] Loss: 1.7115
[Step 31/250] Loss: 1.928
[Step 32/250] Loss: 1.9294
[Step 33/250] Loss: 1.9282
[Step 34/250] Loss: 1.6381
[Step 35/250] Loss: 1.6489
[Step 36/250] Loss: 1.5638
[Step 37/250] Loss: 2.0185
[Step 38/250] Loss: 1.8398
[Step 39/25

Saving model checkpoint to ./qwen-sft-finetuned/checkpoint-250
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attentio

✅ Training complete!


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_posi

✅ LoRA adapters saved to: ./qwen_sft_final


In [6]:
# Save the final fine-tuned model adapters.
final_model_path = "./qwen_sft_final"
trainer.save_model(final_model_path)
print(f"Final model saved to {final_model_path}")

# ================================
# == OPTIMIZATION SUMMARY ==
# No changes were needed. The beauty of LoRA is that you're only saving the "adapter" weights,
# which are tiny (a few megabytes) compared to the multi-gigabyte base model. This step is already
# very resource-friendly.
# ================================

Saving model checkpoint to ./qwen_sft_final
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

Final model saved to ./qwen_sft_final


In [8]:
from transformer_lens import HookedTransformer
import gc
import torch

print("\n" + "="*50)
print("🚀 Running the 'AFTER' benchmark on the fine-tuned model ...")
print("="*50)

# Use the same test_prompt and chat_prompt as before
chat_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": test_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# Generate with fine-tuned LoRA model
inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
generated_ids = model.generate(inputs.input_ids, do_sample=False)
finetuned_output_text = tokenizer.decode(generated_ids[0])

print("\n--- Fine-tuned Model Output ---\n")
print(finetuned_output_text)
print("\n------------------------------\n")

# Run HookedTransformer analysis on the fine-tuned output
hooked_model = HookedTransformer.from_pretrained(model_id)
hooked_model.to("cuda" if torch.cuda.is_available() else "cpu")
hooked_model.eval()

print("Analyzing the fine-tuned generation process...")
with torch.no_grad():
    _, finetuned_cache = hooked_model.run_with_cache(finetuned_output_text)

# Clean up memory
del hooked_model
del finetuned_cache
gc.collect()
torch.cuda.empty_cache()

print("✅ 'AFTER' benchmark complete.")


The following generation flags are not valid and may be ignored: ['top_p'].
- `top_p`: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
If you're using a pretrained model, note that some of these attributes may be set through the model's `generation_config.json` file.



🚀 Running the 'AFTER' benchmark on the fine-tuned model ...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_posi


--- Fine-tuned Model Output ---

<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user

Write a Python program that shows 20 balls bouncing inside a spinning hexagon:
- All balls have the same radius.
- All balls have a number on it from 1 to 20.
- All balls drop from the heptagon center when starting.
- Colors are: #f8b862, #f6ad49, #f39800, #f08300, #ec6d51, #ee7948, #ed6d3d, #ec6800, #ec6800, #ee7800, #eb6238, #ea5506, #ea5506, #eb6101, #e49e61, #e45e32, #e17b34, #dd7a56, #db8449, #d66a35
- The balls should be affected by gravity and friction, and they must bounce off the rotating walls realistically. There should also be collisions between balls.
- The material of all the balls determines that their impact bounce height will not exceed the radius of the heptagon, but higher than ball radius.
- All balls rotate with friction, the numbers on the ball can be used to indicate the spin of the ball.
- The heptagon is spinning around its center, and the speed of s

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_posi

Loaded pretrained model Qwen/Qwen1.5-0.5B-Chat into HookedTransformer
Moving model to device:  cuda
Analyzing the fine-tuned generation process...
✅ 'AFTER' benchmark complete.


In [35]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from transformer_lens import HookedTransformer
import gc

# ====== 1. Paths and Base Model ======
base_model_id = "Qwen/Qwen1.5-0.5B-Chat"   # Base model
peft_model_path = "./qwen_sft_final"       # Your fine-tuned adapter folder

# ====== 2. Load Tokenizer ======
print("📦 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# ====== 3. Load Base Model in 4-bit & Apply LoRA Adapters ======
print("📦 Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    load_in_4bit=True,              # Can change to load_in_8bit=True if preferred
    device_map="auto"
)

print("📦 Loading LoRA adapters...")
model = PeftModel.from_pretrained(model, peft_model_path)
model.eval()

# ====== 4. Test Prompt ======
test_prompt = """
Write a Python program that shows 20 balls bouncing inside a spinning hexagon:
- All balls have the same radius.
- All balls have a number on it from 1 to 20.
- All balls drop from the heptagon center when starting.
- Colors are: #f8b862, #f6ad49, #f39800, #f08300, #ec6d51, #ee7948, #ed6d3d, #ec6800, #ec6800, #ee7800, #eb6238, #ea5506, #ea5506, #eb6101, #e49e61, #e45e32, #e17b34, #dd7a56, #db8449, #d66a35
- The balls should be affected by gravity and friction, and they must bounce off the rotating walls realistically. There should also be collisions between balls.
- The material of all the balls determines that their impact bounce height will not exceed the radius of the heptagon, but higher than ball radius.
- All balls rotate with friction, the numbers on the ball can be used to indicate the spin of the ball.
- The heptagon is spinning around its center, and the speed of spinning is 360 degrees per 5 seconds.
- The heptagon size should be large enough to contain all the balls.
- Do not use the pygame library; implement collision detection algorithms and collision response etc. by yourself. The following Python libraries are allowed: tkinter, math, numpy, dataclasses, typing, sys.
- All codes should be put in a single Python file.
"""

chat_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": test_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# ====== 5. Generate with Large max_new_tokens ======
print("\n🚀 Generating with fine-tuned model...")
inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
generated_ids = model.generate(
    inputs.input_ids,
    do_sample=True,          # Enable sampling for top_p to work
    top_p=0.9,
    temperature=0.7,
    max_new_tokens=10000,     # Increased to allow full code generation
    repetition_penalty=1.1
)
finetuned_output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

print("\n--- Fine-tuned Model Output ---\n")
print(finetuned_output_text)
print("\n------------------------------\n")

# ====== 6. HookedTransformer Analysis ======
print("📊 Loading HookedTransformer for analysis...")
hooked_model = HookedTransformer.from_pretrained(base_model_id)
hooked_model.to("cuda" if torch.cuda.is_available() else "cpu")
hooked_model.eval()

print("🔍 Analyzing the fine-tuned generation process...")
with torch.no_grad():
    _, finetuned_cache = hooked_model.run_with_cache(finetuned_output_text)

print("✅ 'AFTER' benchmark & analysis complete.")


loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/tokenizer_config.json
loading file chat_template.jinja from cache at None


📦 Loading tokenizer...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "fu

📦 Loading base model...


target_dtype {target_dtype} is replaced by `CustomDtype.INT4` for 4-bit BnB quantization
All model checkpoint weights were used when initializing Qwen2ForCausalLM.

All the weights of Qwen2ForCausalLM were initialized from the model checkpoint at Qwen/Qwen1.5-0.5B-Chat.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen2ForCausalLM for predictions without further training.
loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "top_p": 0.8
}



...

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 2816,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_posi


--- Fine-tuned Model Output ---

<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user

Write a Python program that shows 20 balls bouncing inside a spinning hexagon:
- All balls have the same radius.
- All balls have a number on it from 1 to 20.
- All balls drop from the heptagon center when starting.
- Colors are: #f8b862, #f6ad49, #f39800, #f08300, #ec6d51, #ee7948, #ed6d3d, #ec6800, #ec6800, #ee7800, #eb6238, #ea5506, #ea5506, #eb6101, #e49e61, #e45e32, #e17b34, #dd7a56, #db8449, #d66a35
- The balls should be affected by gravity and friction, and they must bounce off the rotating walls realistically. There should also be collisions between balls.
- The material of all the balls determines that their impact bounce height will not exceed the radius of the heptagon, but higher than ball radius.
- All balls rotate with friction, the numbers on the ball can be used to indicate the spin of the ball.
- The heptagon is spinning around its center, and the speed of s

Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645
}

All model checkpoint weights were used when initializing Qwen2ForCausalLM.

All the weights of Qwen2ForCausalLM were initialized from the model checkpoint at Qwen/Qwen1.5-0.5B-Chat.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen2ForCausalLM for predictions without further training.
loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "top_p": 0.8
}

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/4d14e384a4b037942bb3f3016665157c8bcb70ea/vocab.json
loading

Loaded pretrained model Qwen/Qwen1.5-0.5B-Chat into HookedTransformer
Moving model to device:  cuda
🔍 Analyzing the fine-tuned generation process...


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 180.12 MiB is free. Process 19503 has 14.56 GiB memory in use. Of the allocated memory 14.18 GiB is allocated by PyTorch, and 251.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [34]:
del hooked_model
del finetuned_cache
gc.collect()
torch.cuda.empty_cache()

NameError: name 'hooked_model' is not defined